In [0]:
%sql
create or replace temp view vw_dim_customer as(
  with dedup as (
    select
       customer_id
      ,customer_name
      ,row_number() over (
        partition by 
           customer_id 
        order by 
           sale_date desc
          ,ingestion_timestamp desc
      )                                                           as order_by_date
    from workspace.pj_sales.tb_sales_silver
    where part_quantity > 0
  )
  select
     customer_id
    ,customer_name
    ,from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')   as valid_from
    ,null                                                           as valid_to
  from dedup
  where order_by_date = 1
)

In [0]:
%sql
create or replace temp view vw_dim_customer_merge as
select
  vc.*,
  'u' as action
from vw_dim_customer vc -- Aqui o registro serve para ser atualizado
union all
select
  vc.*,
  'i' as action
from vw_dim_customer vc -- Aqui o registro serve para ser inserido

In [0]:
%sql
merge into workspace.pj_sales.tb_dim_customer_gold tc
using vw_dim_customer_merge vcm
on  tc.customer_id = vcm.customer_id
and tc.valid_to is null
and vcm.action = 'u'
when matched and vcm.action = 'u' then
  update set
    tc.valid_to = vcm.valid_from
when not matched and vcm.action = 'i' then
  insert (
     customer_id
    ,customer_name
    ,valid_from
    ,valid_to
  )
  values (
     vcm.customer_id
    ,vcm.customer_name
    ,vcm.valid_from
    ,vcm.valid_to
  )
